# 01 · Machine-translate English SMS → Sinhala & Tamil (NLLB-200)

Makes `data/raw/multilingual_mt.csv` to grow the Sinhala/Tamil **training** data.

* Source = **English rows of `data/processed/train.csv` only** (run `python -m src.data_prep` first), so no test message is ever translated.
* Model = Meta **NLLB-200 distilled 600M** (`sin_Sinh`, `tam_Taml`). Links and numbers are protected so they survive translation.
* `data_prep.py` adds these rows to **TRAIN only**. Hand-written Sri Lankan messages stay in `multilingual_train.csv`.
* Runtime: Colab **GPU (T4)** ≈ 10 min. Afterwards a native speaker should spot-check ~50 rows per language (the `review` sample below).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/smishing-detector
!pip install -q transformers sentencepiece

In [ ]:
import re, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
N_HAM, N_SPAM, N_SMISH = 400, 150, 10_000   # per target language (all smishing rows are used)
MODEL = 'facebook/nllb-200-distilled-600M'
TARGETS = {'si': 'sin_Sinh', 'ta': 'tam_Taml'}

train = pd.read_csv('data/processed/train.csv')
en = train[train.language == 'en']
src = pd.concat([
    en[en.label3 == 'smishing'].sample(min(N_SMISH, (en.label3 == 'smishing').sum()), random_state=SEED),
    en[en.label3 == 'spam'].sample(N_SPAM, random_state=SEED),
    en[en.label3 == 'ham'].sample(N_HAM, random_state=SEED),
]).reset_index(drop=True)
print(src.label3.value_counts())

In [ ]:
# protect links / e-mails / long numbers:  'visit bit.ly/x' -> 'visit ZQX0' -> translate -> put back
PROTECT = re.compile(r'(https?://\S+|www\.\S+|\b[\w-]+\.(?:com|lk|net|org|info|co|uk|xyz|top|buzz|site|online|help|icu|cc|me|ly)(?:/\S*)?|\S+@\S+|\+?\d[\d\s-]{6,}\d)', re.I)

def mask(t):
    found = PROTECT.findall(t)
    for i, f in enumerate(found):
        t = t.replace(f, f'ZQX{i}', 1)
    return t, found

def unmask(t, found):
    for i, f in enumerate(found):
        if f'ZQX{i}' in t: t = t.replace(f'ZQX{i}', f, 1)
        else:             t = t + ' ' + f          # placeholder lost -> append so the link is not dropped
    return re.sub(r'ZQX\d+', '', t).strip()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL, src_lang='eng_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=torch.float16 if dev == 'cuda' else torch.float32).to(dev).eval()

@torch.no_grad()
def translate(texts, tgt, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        batch = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=200).to(dev)
        gen = model.generate(**batch, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), max_new_tokens=256, num_beams=4)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'{tgt}: {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    return out

In [ ]:
SCRIPT = {'si': re.compile(r'[\u0D80-\u0DFF]'), 'ta': re.compile(r'[\u0B80-\u0BFF]')}

def good(t, lang):
    letters = [c for c in t if c.isalpha()]
    if len(letters) < 5: return False
    ratio = sum(bool(SCRIPT[lang].match(c)) for c in letters) / len(letters)
    words = t.split()
    repeat = len(words) > 6 and len(set(words)) / len(words) < 0.4   # NLLB looping
    return ratio >= 0.5 and not repeat

masked = [mask(t) for t in src.text]
rows = []
for lang, code_ in TARGETS.items():
    hyp = translate([m for m, _ in masked], code_)
    for (m, found), h, (_, r) in zip(masked, hyp, src.iterrows()):
        t = unmask(h, found)
        if good(t, lang):
            rows.append(dict(text=t, label=r.label3, language=lang, script='native',
                             category='translated', origin='mt_nllb600m', src_text=r.text))
mt = pd.DataFrame(rows).drop_duplicates('text')
mt.to_csv('data/raw/multilingual_mt.csv', index=False, encoding='utf-8')
print('\nkept', len(mt), 'of', 2 * len(src)); print(pd.crosstab(mt.language, mt.label))

In [ ]:
# native-speaker spot check: 25 per language -> fix in the CSV or delete bad rows
review = mt.groupby('language', group_keys=False).apply(lambda g: g.sample(25, random_state=SEED))
review[['language', 'label', 'src_text', 'text']].to_csv('data/raw/mt_review_sample.csv', index=False, encoding='utf-8')
review[['language', 'label', 'src_text', 'text']].head(10)

In [ ]:
# rebuild train/test - MT rows go to TRAIN only
!python -m src.data_prep
!cat data/processed/data_report.txt